# Churn: who's likely to stop

A classifier that gives each customer (or member, or account) the chance they'll stop: FLAML's AutoML picks the model, it's logged and registered in the project's own MLflow workspace, and once another editor approves it, the project's scoring scores every customer on a schedule.

**To use it:** save a copy (File → Save Notebook As…), then change the parameters in the next cell: the project, the features table and its id column, the query that says who stopped, and the model's name. The examples are the guide's `retail` project; run top to bottom. Data comes through the platform's SQL, as you, at most 100,000 rows: aggregate in SQL, a row per customer.

In [ ]:
import os

PROJECT = os.environ.get("STORSCALE_PROJECT", "retail")   # a scheduled run: the project it runs for                 # the project: its tables, its MLflow workspace, its Models tab
FEATURES = "customer_features"     # the model's inputs, a row per ID: the table the scoring will score
ID = "customer_id"                 # the column naming each row (the scores carry it, so they join back)
COLUMNS = None                     # the inputs, by name; None: every column of FEATURES but ID
LABEL = "churned"                  # 1: they stopped; 0: they didn't. From LABELS:
LABELS = """
SELECT c.customer_id,
       CASE WHEN max(o.ordered_on) >= DATE '2026-09-01' THEN 0 ELSE 1 END AS churned
FROM uploads_customers c
LEFT JOIN uploads_orders o ON o.customer_id = c.customer_id AND NOT o.test AND NOT o.refunded
GROUP BY c.customer_id
"""                                # here: no order since the cutoff, 1 September
MODEL = "churn"                    # the registered model's name
EXPERIMENT = "churn"               # the MLflow experiment its runs go to
TIME_BUDGET = 60                   # seconds FLAML may spend trying models

## The features

**One table, two uses.** A scoring scores a table with the model, so the model learns from one: a row per `ID`, its other columns the model's inputs, as numbers (integers, or `double`: CAST decimals; a 0/1 column per value of a text column), and no label. The label comes from its own query, joined on `ID`. The inputs describe each row **before** the period the label looks at (the cutoff), or the model learns from the answer.

Make the table once, as an editor: adapt the SQL below to your data and run it here (uncomment the last line) or on the project's **SQL** tab.

In [ ]:
FEATURES_SQL = f"""
CREATE OR REPLACE TABLE {FEATURES} AS
SELECT c.customer_id,
       count(o.order_id) AS orders,
       CAST(coalesce(sum(o.amount), 0) AS double) AS revenue,
       date_diff('day', c.signed_up, DATE '2026-09-01') AS tenure_days,
       coalesce(date_diff('day', max(o.ordered_on), DATE '2026-09-01'), 9999) AS days_since_order
FROM uploads_customers c
LEFT JOIN uploads_orders o ON o.customer_id = c.customer_id AND NOT o.test AND NOT o.refunded
                          AND o.ordered_on < DATE '2026-09-01'
GROUP BY c.customer_id, c.signed_up
"""
# import buckets_lake; buckets_lake.sql(PROJECT, FEATURES_SQL)

## The data

The features and the label, joined on `ID`.

In [ ]:
import buckets_lake
import pandas as pd

features = buckets_lake.sql(PROJECT, f"SELECT * FROM {FEATURES}")   # as you, under the project's rules
labels = buckets_lake.sql(PROJECT, LABELS)
data = features.merge(labels[[ID, LABEL]], on=ID)                   # the rows with both
columns = COLUMNS or [c for c in features.columns if c != ID]
text = [c for c in columns if not pd.api.types.is_numeric_dtype(data[c])]
assert not text, f"the model takes numbers: in {FEATURES}, CAST these or make a 0/1 column per value: {text}"
X, y = data[columns], data[LABEL].astype("int64")
print(f"{len(data)} rows, {y.mean():.0%} with {LABEL} = 1; inputs: {', '.join(columns)}")
X.describe()

## Train, log and register

FLAML picks the model by ROC AUC (how well it ranks who stopped above who didn't). The run, its parameters and metrics, and the model are kept in the project's MLflow workspace; `registered_model_name` makes it a new version of `MODEL`.

In [ ]:
import skops.io
from flaml import AutoML
from sklearn.base import clone
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# A quarter held out, to say how well the model ranks rows it hasn't seen.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

mlflow = buckets_lake.mlflow(PROJECT)        # the project's own MLflow workspace, as you
mlflow.set_experiment(EXPERIMENT)
with mlflow.start_run():                     # stamped with you and the catalog's commit
    automl = AutoML()
    # FLAML tries models and settings for TIME_BUDGET seconds. Only the kinds a scoring can
    # load (scikit-learn's, LightGBM's, XGBoost's): the scoring has those, not FLAML.
    automl.fit(X_train, y_train, task="classification", metric="roc_auc", time_budget=TIME_BUDGET,
               estimator_list=["lgbm", "xgb_limitdepth", "rf", "extra_tree", "lrl1"], seed=0, verbose=0,
               mlflow_logging=False)
    best = automl.model.estimator            # the best model itself, without FLAML around it
    auc = roc_auc_score(y_test, best.predict_proba(X_test)[:, 1])
    model = clone(best).fit(X, y)            # the same model and settings, learned again from every row
    mlflow.log_params({"table": FEATURES, "inputs": ",".join(columns), "rows": len(X), "estimator": automl.best_estimator,
                       **{f"best_{k}": v for k, v in automl.best_config.items()}})
    mlflow.log_metrics({"roc_auc": auc, "roc_auc_validation": 1 - automl.best_loss, "positive_rate": float(y.mean())})
    mlflow.sklearn.log_model(model, name="model", registered_model_name=MODEL,
                             pyfunc_predict_fn="predict_proba",   # a scoring takes the last column: the chance of 1
                             input_example=X.head(),              # the signature: the columns, int64 and float64
                             skops_trusted_types=skops.io.get_untrusted_types(data=skops.io.dumps(model)))
print(f"{automl.best_estimator}: ROC AUC {auc:.3f} on the held-out quarter (0.5 is a coin, 1 is perfect)")

The customers most likely to have stopped, by the model:

In [ ]:
data.assign(score=model.predict_proba(X)[:, 1]).sort_values("score", ascending=False).head(10)

## Next

1. **Its card, and an approval.** On the project's **Models** tab, `churn` has a new version, registered by you. **Write its card** (what it's for, the data it learned from, its metrics, its limits), and have another editor of the project approve the version: nobody approves their own. The alias `approved` then names it, and that's the version a scoring loads (`models:/churn@approved`).
2. **Score the table on a schedule.** In the project's spec (**Edit** → JSON), add a scoring:

   ```json
   "scoring": [{"name": "churn-nightly", "model": "churn", "table": "customer_features", "id": "customer_id",
                "output": "customer_churn_scores", "schedule": "0 3 * * *", "drift": 0.2}]
   ```

   If you set `COLUMNS`, add `"columns": [...]` with the same names. Within a minute the **Pipelines** tab lists the scoring; each run, as the project's pipelines, writes `customer_churn_scores` afresh: each row's `customer_id`, its score (the chance of 1), the model's version and when. When an input column drifts past `drift` from the version's first run, the project's alerts say so.
3. **Keep the features current.** A scoring scores the table as it is: rebuild `customer_features` with today as the cutoff (the same SQL, as one of the project's pipelines) before it runs.
4. **Retrain** when the alerts say an input has drifted, or the scores stop matching what happens: run this notebook again, and have the new version approved. Its first scoring run becomes the new reference.